# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [ ]:
%load_ext dotenv
%dotenv ../05_src/.secrets

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [ ]:
# Step 0 : Import environment variables
from pathlib import Path
from pypdf import PdfReader


import getpass
import logging
import os

os.environ["LANGSMITH_TRACING"] = "false"
os.environ["LANGSMITH_API_KEY"] = getpass.getpass()

dirname = Path.cwd()
filename = dirname / "documents/ai_report_2025.pdf"


# Step 1: Setting reader function that will load document from webpage

reader = PdfReader(filename)

if reader:
    logging.info("The document was successfully read.")
else:
    logging.warning("The document was unable to be read.")

# Step 2: Extracting text from PDF and storing it in a document text

document_text=""

for page in reader.pages:

    document_text += page.extract_text() + "\n"

logging.info("All pages have been successful concatenated.")


# Step 3: Confirm the length of the initial document as well as the document_text

print("Document length: ", len(reader.pages))
print("Document text length: ", len(document_text.split(" ")))


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [ ]:
# Step 0: Import all necessary dependencies
import sys
sys.path.append('../05_src')



from langchain.chat_models import init_chat_model
from pydantic import BaseModel, Field
from utils.clients import get_client

MODEL = os.getenv('MODEL', 'gpt-4o-mini')

STRUCTURED_FIELDS={
    "author": "Provide the full name of the author or authors of this document.",
    "title": "Title of the document ",
    "relevance": "A statement, no longer than a paragraph, that explains why this article is relevant for an AI professional development.",
    "summary": "A concise summary no longer than 1000 tokens, yet no shorter than 750 tokens.", # Too short of a summary has insufficient material for evaluation
    "tone": "The tone must be specific and distinguishable. For example,  'Victorian English', 'African-American Vernacular English', 'Formal Academic Writing', 'Bureaucratese'",
    "input": "Total number of input tokens",
    "output": "Total number of output tokens"
    }

PROMPT=f"Process the following document: {document_text}"

client = get_client()

# Step 1: Load the model 

llm = init_chat_model(
    model=MODEL,
    model_provider="openai",
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
)


# Step 2: Define the response output

class Metadata(BaseModel):
    author: str=Field(STRUCTURED_FIELDS["author"])
    title: str=Field(STRUCTURED_FIELDS["title"])
    relevance: str=Field(STRUCTURED_FIELDS["relevance"])
    summary: str=Field(STRUCTURED_FIELDS["summary"])
    tone: str=Field(STRUCTURED_FIELDS["tone"])
    input_tokens: str=Field(STRUCTURED_FIELDS["input"])
    output_tokens:str=Field(STRUCTURED_FIELDS["output"])

structured_llm=llm.with_structured_output(Metadata)


# Step 3: Submitting request
metadata = structured_llm.invoke(PROMPT)

# Step 4: Post-processing request


# Step 5: Print output

print(metadata)


""" OUTPUT ANALYSIS

author='MIT NANDA Team' 
title='The GenAI Divide: State of AI in Business 2025' 
relevance='High - provides insights on the impact and challenges of AI adoption in organizations.' 
summary="The report discusses the vast investments in Generative AI (GenAI) and the poor returns on these investments for most organizations, 
highlighting the disparity between high adoption rates and low transformation. It identifies key barriers to scaling successful AI implementations, 
notes the emergence of 'shadow AI' as employees utilize personal tools independently, and outlines strategies for successful AI deployment. 
The document also introduces trends and patterns that separate successful organizations from those stalled in pilot phases." 
tone='Analytical and informative, focusing on data and findings.' 
input=2852
output=672

The authors for this paper were INCORRECTLY identified. 
The title was CORRECTLY identified✅. 
The relevance was CORRECTLY identified✅. 
The summary was POOR and not in the correct tone. 
The tone was correctly identified✅.
The input token were CORRECTLY calculated.
The output token were CORRECTLY calculated✅. 

Conclusion (4/6)


"""

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [ ]:
# Step 0: Import all necessary modules

from deepeval import evaluate
from deepeval.test_case import LLMTestCase
from deepeval.metrics import SummarizationMetric
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams
from deepeval.models import GPTModel

from typing import Dict, List


model = GPTModel(
        model=MODEL,
        temperature=1,
        api_key='any value',
        default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
        base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
    )

# Step 1: Defining important constant

""""

The assessment questions were generated by chatGPT 5.5 instant in the spirit of AI evaluation. 
It is important in such methodology to also list what each question is supposed to test for transaparency concerned.

"""

ASSESSMENT_QUESTIONS={
    "questions": [
        "What is the 'GenAI Divide,' and what evidence does the report use to argue that it exists?",
        "According to the report, why do most enterprise AI pilots fail to reach production, and why is the problem not primarily about model quality, regulation, or infrastructure?",
        "How does the report distinguish between the success of general-purpose tools like ChatGPT and the poor performance of many enterprise AI solutions?",
        "What characteristics do the report's most successful AI buyers and vendors share, and how do these differ from organizations that remain stuck in pilot mode?",
        "Where does the report argue organizations are overlooking the greatest AI return on investment, and what evidence does it provide?"
        ],
    "concepts_tested": [
        "Whether the summary captures the report's core thesis (not just that AI adoption is high, but that value creation is highly uneven, including the 95% vs. 5% finding).",
        "Whether the summary explains the report's central causal argument—that the key barrier is the 'learning gap' (systems that don't learn, remember context, or adapt to workflows).",
        "Whether the summary captures one of the report's key paradoxes: widespread adoption of consumer AI alongside disappointing enterprise deployments, and the reasons behind it.",
        "Whether the summary includes the report's practical recommendations, such as workflow-specific customization, trusted partnerships, operational metrics, co-evolution with vendors, and bottom-up adoption.",
        "Whether the summary includes the counterintuitive finding that back-office automation (e.g., BPO replacement, finance, document processing) often delivers higher ROI than the sales and marketing use cases that receive most AI investment."
        ]

}


"""

Here is an explanation for the coherence, tone, and safety metric instructions

Safety: Referring to the class slide 3, evaluation, the conditions for safety were established to be 
the level of hallucinations and the possibility of bias.

"""

ASSESSEMENT_METRICS={
    "coherence": {"name": "coherence", "criteria": [
        "Determine whether the output is coherent with the provided input.",
        "Give a rating out of 10 on the coherence of the output."
        "Explain your reasonning for that score under 1000 tokens."]},
    "tone": {"name": "tone", "criteria": ["Provide a faithful description of the tone use in the document."]},
    "safety": {"name": "safety", "criteria": ["Review of the safetiness of the output. Be especially regarding of the level of hallucination and the possible biases in the output."]}
}


# Step 2: Building Evaluation Reusable Functions

baseline_comparison=LLMTestCase(input=document_text, actual_output=metadata.summary)

class ScoringMetrics():
    
    def __init__(self, input, actual_output, test_case):

        self.input = input
        self.actual_output = actual_output
        self.baseline_comparison=test_case

        pass

    def single_criteria_score(self, name: str, criteria: List[str]):
        metrics=GEval(
            name=name,
            criteria=criteria[0] if len(criteria) == 1 else None,
            evaluation_steps=criteria if len(criteria) > 1 else None,
            evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
            model=model
            )
        
        results = evaluate(test_cases=[baseline_comparison], metrics=[metrics])

        return results

    def summary_score(self):
        metrics=SummarizationMetric(
        threshold=0.5,
        model=model,
        assessment_questions=ASSESSMENT_QUESTIONS["questions"]
    )
        results = evaluate(test_cases=[baseline_comparison], metrics=[metrics])

        return results


    def format_score(self, result) -> Dict[str, str]:
        extracted_results = result.test_results[0].metrics_data[0].model_dump()

        score = extracted_results["score"]
        reason = extracted_results["reason"]

        formatted = {"score": score, "reason": reason}

        return formatted


    def build_scores(self, summary: Dict[str, str], coherence: Dict[str, str], tone: Dict[str, str], safety: Dict[str, str]) -> Dict[str, str]:
        score_dict = {
            "SummarizationScore": summary["score"],
            "SummarizationReason": summary["reason"],
            "CoherenceScore": coherence["score"], 
            "CoherenceReason": coherence["reason"],
            "ToneScore": tone["score"],
            "ToneReason": tone["reason"],
            "SafetyScore": safety["score"], 
            "SafetyReason": safety["reason"],

        }
        return score_dict
    

# Step 3: Initializing Scoring Metrics

scoring = ScoringMetrics(document_text, metadata.summary, baseline_comparison)



# Step 5: Evaluating all metrics

sumarization_results =  scoring.summary_score()
coherence_results = scoring.single_criteria_score(ASSESSEMENT_METRICS["coherence"]["name"], ASSESSEMENT_METRICS["coherence"]["criteria"])
tone_results = scoring.single_criteria_score(ASSESSEMENT_METRICS["tone"]["name"], ASSESSEMENT_METRICS["tone"]["criteria"])
safety_results = scoring.single_criteria_score(ASSESSEMENT_METRICS["safety"]["name"], ASSESSEMENT_METRICS["safety"]["criteria"])



# Step 6: Building Scores Report

report = scoring.build_scores(scoring.format_score(sumarization_results), scoring.format_score(coherence_results), scoring.format_score(tone_results), scoring.format_score(safety_results))

print("Report: ", report)


""" OUTPUT ANALYSIS


Report:  {
'SummarizationScore': 0.0, 
'SummarizationReason': 'The score is 0.00 because the summary contradicts essential points made in the original text regarding barriers to GenAI success and introduces 
unrelated concepts that aren’t present in the original text. Additionally, there are multiple questions that the summary fails to address, demonstrating a significant 
lack of alignment with the source material.', 
'CoherenceScore': 0.8562176500885798, 
'CoherenceReason': 'The output effectively summarizes the key themes and findings of 
the provided report, particularly the GenAI Divide, barriers to implementation, and the approaches of successful organizations. It captures essential details like the 95% 
failure rate and the importance of adaptability in AI tools. However, while it is coherent and addresses the main points well, it could benefit from a more detailed 
exploration of specific metrics or examples presented in the report to enhance the depth of analysis.', 
'ToneScore': 0.7960571954194481, 
'ToneReason': "The response effectively captures the core themes of the input document, particularly the GenAI Divide and the barriers that organizations face in implementing 
AI solutions. It accurately reflects the emotional intensity of frustration regarding the lack of returns on investments and highlights the importance of adaptable systems and 
partnerships. However, it could provide more specific examples or elaboration on the learning gaps mentioned, which would strengthen its alignment with the input's detailed
 analysis.", 
 'SafetyScore': 0.802665172732539, 
 'SafetyReason': 'The response accurately summarizes key aspects of the input, such as the GenAI Divide, the lack of return on investment, and the barriers related to learning 
 gaps and poor integration. It effectively captures the main themes of the report, including the strategies of successful organizations and the shift towards adaptive AI systems.
However, it could improve by providing more specific examples or details from the original text to further enhance clarity and depth.'}

Control:
'SummarizationScore': 0.0, 
'CoherenceScore': 0.8562176500885798,
'ToneScore': 0.7960571954194481,
'SafetyScore': 0.802665172732539, 


"""




# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [ ]:

# Step 1: Building a complete prompt


ROLEPLAY="You are an experienced AI tech journalist and editor. You are well-versed in current topic in AI and their impact on society. Your job is to produce a summary of a " \
"technical and accurate summary that explains why the provided document is relevant to an AI professional and for their professional development." \
" You must show discernment and only select the most relevant information.  "
FEW_SHOTS="""

Summary 1: Stanford AI Index 2025 – Why Every AI Professional Should Read It

The Stanford AI Index 2025 provides one of the most comprehensive annual snapshots of the global AI landscape. Rather than focusing on a single technology or product, it combines data from research, industry, education, government, investment, and policy to explain where artificial intelligence is today and where it is heading. For AI professionals, the report is valuable because it provides context that goes beyond technical developments. It helps practitioners understand the broader ecosystem in which AI systems are built, deployed, regulated, and adopted.
One of the report's central messages is that AI continues to improve rapidly while becoming increasingly accessible. Foundation models are becoming more capable across reasoning, coding, mathematics, and multimodal tasks. At the same time, the cost of developing and deploying AI models is changing, with open-source models narrowing the performance gap with proprietary systems. This means that competitive advantage is shifting away from simply having access to powerful models and toward how organizations integrate AI into products, workflows, and decision-making.
The report also highlights that enterprise adoption continues to grow, but successful implementation depends less on the underlying model than on organizational readiness. Companies that invest in data quality, governance, employee training, and clear business objectives are more likely to realize measurable value from AI initiatives. This reinforces the idea that AI projects are as much organizational transformations as they are technical implementations.
Another important theme is the growing attention to responsible AI. Governments around the world are introducing new regulations, while organizations are increasing investments in model evaluation, transparency, security, and risk management. AI professionals therefore need a working knowledge of governance, privacy, bias mitigation, and evaluation methodologies in addition to traditional machine learning skills.
For professional development, the report suggests that successful AI practitioners should cultivate a broad skill set. Technical expertise remains essential, but it should be complemented by business understanding, communication skills, data literacy, and an awareness of emerging policy and ethical considerations. Professionals who can bridge technical and organizational perspectives will be well positioned to lead AI initiatives.
The key takeaway is that the AI landscape is evolving from experimentation to operational maturity. Long-term success will come from combining strong technical foundations with the ability to deliver reliable, trustworthy, and business-focused AI solutions. AI professionals who continually monitor industry trends, expand their interdisciplinary knowledge, and adapt to changing technologies will be better prepared to create lasting value in an increasingly competitive field.

Summary 2: McKinsey – Superagency in the Workplace: Empowering People to Unlock AI's Full Potential

McKinsey's Superagency in the Workplace argues that the next wave of AI value will come not from replacing people, but from enabling them to perform at a higher level. The report introduces the concept of "superagency," where employees use AI as an intelligent collaborator that amplifies creativity, productivity, and decision-making. For AI professionals, the report emphasizes that success depends less on deploying advanced models and more on designing systems that people trust, adopt, and integrate into their daily work.
A key insight is that many organizations have moved beyond experimenting with AI but still struggle to scale its impact. Technical capabilities are advancing rapidly, yet organizational adoption often lags because employees lack confidence, training, or clear guidance on how AI should fit into existing workflows. The report suggests that successful AI transformation requires leadership support, practical training, and a culture that encourages experimentation while maintaining accountability.
The report also highlights that AI changes the nature of work rather than simply automating tasks. Routine activities such as drafting documents, summarizing information, generating code, or analyzing large datasets can increasingly be delegated to AI systems. This allows professionals to spend more time on strategic thinking, problem solving, customer engagement, and innovation. As a result, the most valuable employees will be those who know how to effectively direct AI systems, evaluate their outputs, and combine machine-generated insights with human judgment.
Another important theme is the growing importance of AI literacy across the workforce. Organizations should not limit AI expertise to technical teams. Managers, analysts, designers, marketers, and frontline employees all benefit from understanding AI's capabilities, limitations, and appropriate use cases. AI professionals therefore have an important role as educators, helping colleagues develop confidence and responsible practices when working with AI tools.
For professional development, the report encourages AI practitioners to expand beyond technical implementation. Skills such as change management, communication, prompt engineering, workflow design, evaluation, and human-centered design are becoming increasingly valuable. Professionals who can translate business challenges into AI-enabled solutions—and help teams adopt those solutions successfully—will create greater organizational impact than those focused solely on model development.
The report's central message is that AI should be viewed as a force multiplier for human capability rather than a replacement for expertise. Organizations that invest equally in technology, people, and organizational change are more likely to achieve sustainable AI adoption. For AI professionals, this means developing not only technical excellence but also leadership, collaboration, and the ability to build trust between people and intelligent systems. These capabilities will become defining characteristics of successful AI leaders as AI becomes embedded across every function of the enterprise.

Summary 3: AI Report 2025 – Closing the Enterprise AI Gap

The AI Report 2025 explores why some organizations achieve significant business value from generative AI while many others remain stuck in pilot projects or isolated experiments. Its central argument is that the challenge facing most enterprises is no longer access to powerful AI models. Instead, the greatest barrier is turning those models into systems that consistently improve business processes and produce measurable outcomes. For AI professionals, the report shifts the conversation from model performance to implementation, organizational learning, and operational excellence.
A major theme is the emergence of the "GenAI Divide," where a relatively small group of organizations captures most of the value generated by AI while many others struggle to move beyond experimentation. The report argues that successful organizations treat AI as a business capability rather than a standalone technology project. They invest in workflow integration, governance, employee training, and continuous improvement instead of focusing exclusively on selecting the latest language model.
The report also explains why enterprise AI often underperforms despite impressive demonstrations of generative AI capabilities. General-purpose AI systems are excellent at answering questions, generating text, and supporting creative work, but enterprise environments require additional capabilities such as maintaining organizational context, integrating with existing systems, preserving institutional knowledge, and adapting to changing business processes. Without these supporting components, AI applications frequently produce inconsistent results that reduce user confidence and limit adoption.
Another important finding is that organizations often underestimate the importance of operational workflows. High-value AI deployments are typically built around specific business problems rather than broad, open-ended use cases. Functions such as document processing, customer support, finance, compliance, procurement, and business process outsourcing often generate stronger and more measurable returns than highly visible marketing or innovation initiatives. The report encourages organizations to prioritize use cases where AI can deliver consistent improvements in productivity, quality, or decision-making.
For AI professionals, the report reinforces the importance of developing skills that extend beyond model development. Success increasingly depends on understanding business operations, designing reliable workflows, evaluating system performance, managing organizational change, and collaborating with subject matter experts. Building trustworthy AI systems requires attention to data quality, retrieval strategies, human oversight, evaluation frameworks, and continuous monitoring after deployment.
The report concludes that enterprise AI is entering a new phase of maturity. Competitive advantage will increasingly come from an organization's ability to integrate AI into everyday operations, continuously improve workflows, and measure business impact over time. AI professionals who combine technical expertise with business knowledge, systems thinking, and implementation skills will be best positioned to lead this transition. Rather than viewing AI as a collection of impressive models, the report encourages practitioners to think of AI as a long-term organizational capability that requires ongoing learning, adaptation, and collaboration to generate sustainable value.

"""
REQUIREMENTS="""
1. The token usage of the summary is limited at 1000, yet should not be lower than 750 tokens.
2. Be specific when evaluating stakeholders and consequences on communities.
3. Mention technical details and processes when necessary. """

TONE="Use the African American Vernacular English tone in your response."
CONTEXT=document_text
QUESTION="Using the provided document, correctly identify the authors, the title, and the tone. Generate a summary that answer why this document is relevant to AI professional."

"""

I separated prompts in multiple steps that have previously shown to (from class material).
I included reuquirements to avoid reitering errors that have been observed using BaseModel (from the previous cells)


"""

def system_prompt() -> str:

    structure=f"""
    {ROLEPLAY}

    Important rules to follow:
    {REQUIREMENTS}

    Tone: 
    {TONE}

    Examples:
    {FEW_SHOTS}
    """
    return structure

def user_prompt() -> str:

    structure=f"""
    Context: {CONTEXT}
    Question: {QUESTION}
    """
    return structure



improved_prompt = [
        {"role": "system", "content": system_prompt()},
        {"role": "user", "content": user_prompt()}
        ]


# Step 2: Generating the response


structured_llm=llm.with_structured_output(Metadata)

improved_metadata=structured_llm.invoke(improved_prompt)

print("Improved Metadata: ", improved_metadata)


# Step 3: Evaluating the response

baseline_comparison=LLMTestCase(input=document_text, actual_output=improved_metadata.summary)

scoring = ScoringMetrics(CONTEXT, improved_metadata.summary, baseline_comparison)

# Step 5: Evaluating all metrics

sumarization_results =  scoring.summary_score()
coherence_results = scoring.single_criteria_score(ASSESSEMENT_METRICS["coherence"]["name"], ASSESSEMENT_METRICS["coherence"]["criteria"])
tone_results = scoring.single_criteria_score(ASSESSEMENT_METRICS["tone"]["name"], ASSESSEMENT_METRICS["tone"]["criteria"])
safety_results = scoring.single_criteria_score(ASSESSEMENT_METRICS["safety"]["name"], ASSESSEMENT_METRICS["safety"]["criteria"])



# Step 6: Building Scores Report

report = scoring.build_scores(scoring.format_score(sumarization_results), scoring.format_score(coherence_results), scoring.format_score(tone_results), scoring.format_score(safety_results))

print("Report: ", report)


""" OUTPUT ANALYSIS

 author='MIT NANDA, Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari' 
 title='The GenAI Divide: State of AI in Business 2025' 
 relevance='This document is critical for AI professionals as it unpacks the substantial gap between AI adoption and effective implementation. 
    By revealing stark disparities in enterprise AI deployment and effective generative AI (GenAI) systems, it informs practitioners about the barriers
    they face while navigating the landscape of AI in business today. Furthermore, the insights into investment biases, user preferences between 
    consumer-grade and enterprise systems, and organizational structures that facilitate or hinder success are particularly useful for AI professionals
    seeking to enhance their strategic positioning within organizations.' 
 summary="Man, let me break it down for ya. The ‘GenAI Divide’ report from MIT NANDA’s fam talks about how AI in companies is like a rollercoaster
   – you got all this hype and investment, with billions plowed into GenAI, but a whopping 95% of organizations ain't seein' nothin' in return. 
   Basically, this report puts the spotlight on how while everybody's playing with cool tools like ChatGPT, most folk still ain't crossing over to actually see real 
   transformations in their businesses. \n\nThis is mad important for AI professionals because it gives 'em the lowdown on why some companies are catching the wave 
   while others are just bobbing at sea with no real change. It digs deep into what they call the 'learning gap' that’s keeping organizations stuck – tools that can’t 
   learn or adapt just ain't getting the job done. They highlight how these rigid systems often fail to integrate well into the workflows people already have, making it
     hard for companies to realize the potential of these fancy AI tools.\n\nYou know what's wild? It ain't about budget or the right tools. The paper points out that the 
     heart of the problem is that most GenAI solutions don’t adapt or remember feedback. So, while everyone and their mama is trying to get all this AI going, they end up 
     stuck in pilot projects that go nowhere. The report suggests that organizations that get real value from AI tools invest in systems that learn over time and fit snugly 
     into existing processes.\n\nFor AI professionals, this info is gold. It sheds light on what buyers really want: AI systems that don’t just spit out data but actually 
     evolve and improve over time. The report emphasizes that those who cross the GenAI Divide successfully are likely the ones who focus on partnerships with vendors, 
     integrate tools deeply into their workflows, and foster a culture of constant adaptation and learning.\n\nFinally, the report talks about the importance of 
     understanding organizational design so firms can effectively utilize AI. Those making the most successful transitions are the ones letting their line managers 
     drive changes and really engaging with the actual users of the AI tools. \n\nSo if you’re in the AI game, you gotta understand these patterns, the mistakes many 
     are making, and how to leverage partnerships to get on the right side of the GenAI Divide. This document is your roadmap to navigating these waters and coming out 
     on top." 
     tone='African American Vernacular English' 
     input_tokens='3263' 
     output_tokens='975'


The authors for this paper were ALL CORRECT identified✅. 
The title was CORRECTLY identified✅. 
The relevance was CORRECTLY identified✅. 
The summary was GREAT and in the correct tone✅. 
The tone was correctly identified✅.
The input token were CORRECTLY calculated.
The output token were CORRECTLY calculated✅. 

Conclusion (6/6)

Control vs Improved :

'SummarizationScore': 0.0 vs 0.16 ✅

There is definitely an improvement. The summary could be more accurate if more tokens were allowed. Possibly, the questions were too complex to gauge the quality of the summaries.   

'CoherenceScore': 0.8562176500885798 vs 0.87 ✅,
'ToneScore': 0.7960571954194481 vs 0.84 ✅,
'SafetyScore': 0.802665172732539 vs 0.85 ✅, 


Overall the improved version obtained better results than the control version.

"""


Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
